In [1]:
# Import core data handling libraries
import pandas as pd
import numpy as np

In [2]:
# Import regular expressions for light text cleaning if needed
import re

# Import PyTorch
import torch
import torch.nn as nn

# Import train test split
from sklearn.model_selection import train_test_split

# Import class weight utility
from sklearn.utils.class_weight import compute_class_weight

# Import Hugging Face dataset utility
from datasets import Dataset

# Import tokenizer, model, training arguments, and trainer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# Import model saving utility
import joblib

In [46]:
# Import evaluation metrics
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [3]:
# Optional display settings for easier dataframe viewing
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

In [4]:
# Load the 500k sampled dataset from csv
df = pd.read_csv(r"../data/processed/cfpb_sample_500k.csv")

In [5]:
df.shape

(500000, 4)

In [6]:
# Check column names
df.columns.tolist()

['text', 'target', 'text_clean', 'target_grouped']

In [7]:
df

,text,target,text_clean,target_grouped
0,Hi I am submitting this XXXX XXXX this isn't any influence and this is not a third party. XXXX has low and unfair credit number for me in their report. I have 87complained times. The problem has n...,Credit reporting,hi i am submitting this xxxx xxxx this isn t any influence and this is not a third party xxxx has low and unfair credit number for me in their report i have 87complained times the problem has not ...,Credit reporting
1,"A federal education loan was fraudulently opened in my name. I filed an FTC Identity Theft Report, a DOE Loan Forgery Discharge Application, and submitted supporting documents. Despite this, the D...",Debt collection,a federal education loan was fraudulently opened in my name i filed an ftc identity theft report a doe loan forgery discharge application and submitted supporting documents despite this the depart...,Debt collection
2,"I am writing to have the following information removed from my credit file, the items that I need deleted are going to be attached in a word document. I am a victim of identity theft. I have multi...",Credit reporting,i am writing to have the following information removed from my credit file the items that i need deleted are going to be attached in a word document i am a victim of identity theft i have multiple...,Credit reporting
3,"I want this late payment remove from my account, and I never been late with this account I working so hard to pay this monthly and this is not acceptable. \nSee the documents attached.",Credit card,i want this late payment remove from my account and i never been late with this account i working so hard to pay this monthly and this is not acceptable see the documents attached,Credit card
4,I am disputing several items on my Experian credit report that appear to be inaccurate or incomplete. The details of the accounts in question are as follows : 1. Charged-Off Accounts XXXX XXXX XXX...,Credit reporting,i am disputing several items on my experian credit report that appear to be inaccurate or incomplete the details of the accounts in question are as follows 1 charged off accounts xxxx xxxx xxxx xx...,Credit reporting
...,...,...,...,...
499995,XXXX should not be reporting to XXXX ; XXXX and Transunion that I am ( 30 ) days late on the XXXX partial account number XXXX. ( Please see page attached from my credit report. ) This account had...,Credit reporting,xxxx should not be reporting to xxxx xxxx and transunion that i am 30 days late on the xxxx partial account number xxxx please see page attached from my credit report this account had been transfe...,Credit reporting
499996,The credit bureaus are reporting inaccurate/outdated/incomplete personal information.,Credit reporting,the credit bureaus are reporting inaccurate outdated incomplete personal information,Credit reporting
499997,"We receive at least 3 calls an hour, all day long, all hours of the day, even weekends. Have asked them to stop and they have not. The dates have been ongoing for weeks. Todays date, XX/XX/19 was ...",Debt collection,we receive at least 3 calls an hour all day long all hours of the day even weekends have asked them to stop and they have not the dates have been ongoing for weeks todays date xx xx 19 was the las...,Debt collection
499998,"These can be combined On my credit report, you have shown incorrect accounts that should not be there at all. This is not only unjust to me, but it's also concerning, because I've never done or ma...",Credit reporting,these can be combined on my credit report you have shown incorrect accounts that should not be there at all this is not only unjust to me but it s also concerning because i ve never done or made a...,Credit reporting


In [8]:
df.columns

Index(['text', 'target', 'text_clean', 'target_grouped'], dtype='object')

In [9]:
# create labels to id and id to labels
label2id = {label: i for i, label in enumerate(sorted(df['target_grouped'].unique()))}
id2label = {i:label for label, i in label2id.items()}

In [10]:
# create numeric label columns

df["label"] = df["target_grouped"].map(label2id)

In [11]:
label2id

{'Auto loan': 0,
 'Bank account': 1,
 'Credit card': 2,
 'Credit reporting': 3,
 'Debt collection': 4,
 'Money transfer': 5,
 'Mortgage': 6,
 'Other': 7,
 'Personal loan': 8,
 'Student loan': 9}

In [12]:
id2label

{0: 'Auto loan',
 1: 'Bank account',
 2: 'Credit card',
 3: 'Credit reporting',
 4: 'Debt collection',
 5: 'Money transfer',
 6: 'Mortgage',
 7: 'Other',
 8: 'Personal loan',
 9: 'Student loan'}

In [13]:
df[["text_clean", "target_grouped", "label"]]

,text_clean,target_grouped,label
0,hi i am submitting this xxxx xxxx this isn t any influence and this is not a third party xxxx has low and unfair credit number for me in their report i have 87complained times the problem has not ...,Credit reporting,3
1,a federal education loan was fraudulently opened in my name i filed an ftc identity theft report a doe loan forgery discharge application and submitted supporting documents despite this the depart...,Debt collection,4
2,i am writing to have the following information removed from my credit file the items that i need deleted are going to be attached in a word document i am a victim of identity theft i have multiple...,Credit reporting,3
3,i want this late payment remove from my account and i never been late with this account i working so hard to pay this monthly and this is not acceptable see the documents attached,Credit card,2
4,i am disputing several items on my experian credit report that appear to be inaccurate or incomplete the details of the accounts in question are as follows 1 charged off accounts xxxx xxxx xxxx xx...,Credit reporting,3
...,...,...,...
499995,xxxx should not be reporting to xxxx xxxx and transunion that i am 30 days late on the xxxx partial account number xxxx please see page attached from my credit report this account had been transfe...,Credit reporting,3
499996,the credit bureaus are reporting inaccurate outdated incomplete personal information,Credit reporting,3
499997,we receive at least 3 calls an hour all day long all hours of the day even weekends have asked them to stop and they have not the dates have been ongoing for weeks todays date xx xx 19 was the las...,Debt collection,4
499998,these can be combined on my credit report you have shown incorrect accounts that should not be there at all this is not only unjust to me but it s also concerning because i ve never done or made a...,Credit reporting,3


### Sample data for stronger transformation training

In [14]:
df_transformer = df[["text_clean", "target_grouped", "label"]].sample(
    n=250000,
    random_state = 42
)

In [15]:
df_transformer.isnull().sum()

text_clean        0
target_grouped    0
label             0
dtype: int64

In [16]:
df_transformer.shape

(250000, 3)

In [17]:
df_transformer["target_grouped"].value_counts()

target_grouped
Credit reporting    167210
Debt collection      27473
Credit card          15355
Bank account         12472
Mortgage              9394
Money transfer        7661
Student loan          3991
Auto loan             3223
Personal loan         2893
Other                  328
Name: count, dtype: int64

### Create Train-Test-Split

In [18]:
# split data in test and train

train_df, test_df = train_test_split(
    df_transformer,
    test_size = 0.2,
    random_state=42,
    stratify=df_transformer["label"]
)

In [19]:
# check shape

print(train_df.shape, test_df.shape)

(200000, 3) (50000, 3)


In [20]:
# check label distribution in train split

train_df["target_grouped"].value_counts()

target_grouped
Credit reporting    133768
Debt collection      21979
Credit card          12284
Bank account          9978
Mortgage              7515
Money transfer        6129
Student loan          3193
Auto loan             2578
Personal loan         2314
Other                  262
Name: count, dtype: int64

In [21]:
# check label distribution in test split

test_df["target_grouped"].value_counts()

target_grouped
Credit reporting    33442
Debt collection      5494
Credit card          3071
Bank account         2494
Mortgage             1879
Money transfer       1532
Student loan          798
Auto loan             645
Personal loan         579
Other                  66
Name: count, dtype: int64

### Convert to transformers dataset

In [22]:
# convert train and test dataframes to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df[["text_clean", "label"]])
test_dataset = Dataset.from_pandas(test_df[["text_clean", "label"]])

In [23]:
# check names

print(train_dataset.column_names)

['text_clean', 'label', '__index_level_0__']


In [24]:
print(test_dataset.column_names)

['text_clean', 'label', '__index_level_0__']


### set the model name

[Model Link](https://huggingface.co/FacebookAI/roberta-base)

In [25]:
# set the stronger transformer backbone
model_name = "roberta-base"

In [26]:
# Load Tokenizer for RoBERTa
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [27]:
# create the tokenization function

def tokenize_function(examples):
    return tokenizer(
        examples["text_clean"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

#### Apply tokenizer

In [28]:
# apply tokenization to train and test dataset

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batch_size=True)

Map:   0%|          | 0/200000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [29]:
train_dataset.column_names

['text_clean', 'label', '__index_level_0__', 'input_ids', 'attention_mask']

In [30]:
# removed unused columns

cols_to_remove = [col for col in ["text_clean", "__index_level_0__"] if col in train_dataset.column_names]
train_dataset = train_dataset.remove_columns(cols_to_remove)
test_dataset = test_dataset.remove_columns(cols_to_remove)

In [31]:
# set pytorch format for pytorch
train_dataset.set_format("torch")
test_dataset.set_format("torch")

In [32]:
train_dataset

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 200000
})

In [33]:
# check final dataset columns and one sample

print(train_dataset.column_names)
print(train_dataset[0])

['label', 'input_ids', 'attention_mask']
{'label': tensor(9), 'input_ids': tensor([    0,   118,   475,   145,  1340,    13,    10,  2541,    77,   939,
          174,     5,  2737,  3023, 46628,  3023, 46628,  1564,    14,   939,
           74,    45,    28,   441,     7,  2725,   528,     7,   474,  1486,
            8,   939,  1882,    66,     9,     5,   586,   648,    51,   439,
          789,     8, 12069,     5,  2541,    25,   114,   939,   393,  2294,
         5190,   939,    21,  4768,    25,  1085,   115,    28,   626,    53,
           89,  1302,     7,    28,   402,     2,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,    

### Compute class weights from the training splits

In [34]:
# get sorted label ids from the training data

class_ids = sorted(train_df["label"].unique())

In [36]:
class_ids

[np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(9)]

In [35]:
# compute balanced class weights

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(class_ids),
    y=train_df["label"].values
)

In [37]:
class_weights

array([ 7.7579519 ,  2.0044097 ,  1.62813416,  0.14951259,  0.90995951,
        3.26317507,  2.66134398, 76.33587786,  8.64304235,  6.26370185])

In [38]:
# convert class weights to Pytorch tensor

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

In [39]:
class_weights_tensor

tensor([ 7.7580,  2.0044,  1.6281,  0.1495,  0.9100,  3.2632,  2.6613, 76.3359,
         8.6430,  6.2637])

#### Load RoBERTa model

In [41]:
# Load roberta for sequence classification

model=AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

c:\Users\tevin\OneDrive\Desktop\cfpb-complaint-intelligence\venv\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tevin\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#### Create a weighted Trainer class

In [58]:
# Create a custom Trainer that uses weighted cross-entropy loss
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor.to(model.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

#### Create the metric function

In [47]:
# create metric function for evaluation

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    accuarcy = accuracy_score(labels, preds)
    weighted_f1 = f1_score(labels, preds, average="weighted", zero_division=0)
    macro_f1 = f1_score(labels, preds, average=macro, zero_division=0)

    return {
        "accuracy": accuarcy,
        "weighted_f1": weighted_f1,
        "macro_f1": macro_f1
    }

### Training Arguments and Trainer setup

In [48]:
# define training arguments for RoBERTa

training_args = TrainingArguments(
    output_dir="./roberta_results_250k",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="weighted_f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none"
)

In [59]:
# Recreate weighted trainer for RoBERTa
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

#### device check

In [50]:
# check whether training will use GPU or CPU

print("CUDA available: ", torch.cuda.is_available())
print("Device: ", model.device)

CUDA available:  False
Device:  cpu


### Train Model

In [ ]:
# Train the RoBERTa model

trainer.train()

Epoch,Training Loss,Validation Loss
